# Chapter 4: Deep RL — Case Study

## DQN vs REINFORCE vs Actor-Critic on CartPole

This notebook compares three deep RL algorithms from Chapter 4 on the
CartPole-v1 environment using the `rlbook` library. We train each algorithm
across multiple seeds and compare learning curves, final performance, and
sample efficiency.

In [ ]:
# Setup — run once
# Local:  pip install -e '.[deep]'
# Colab:  !pip install -q "git+https://github.com/rl-textbook/reinforcement-learning.git#egg=rlbook[deep]"

import os, json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn.functional as F

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

from rlbook.environments import CartPoleWrapper, set_global_seed
from rlbook.algorithms.deep import (
    PolicyNet, QNet, Critic,
    RunConfig, run_reinforce, run_actor_critic, run_dqn, DQNConfig,
)

## 1. Environment

In [ ]:
env = CartPoleWrapper(seed=42)
print(f"Observation dim: {env.obs_dim}, Actions: {env.n_actions}")
print(f"CartPole-v1: solved when mean return ≥ 475 over 100 episodes")

## 2. Training All Three Algorithms

We train each algorithm across 3 seeds for statistical robustness.
Evaluation happens every 2000 environment steps.

In [ ]:
LOGDIR = 'logs'
os.makedirs(LOGDIR, exist_ok=True)

cfg = RunConfig(max_steps=100_000, eval_every=2000, eval_episodes=5)
seeds = [42, 123, 456]

all_logs = []

for seed in seeds:
    print(f'\n--- Seed {seed} ---')
    env = CartPoleWrapper(seed=seed)

    # REINFORCE
    logs_r = run_reinforce(seed, env, cfg, lr=1e-3)
    all_logs.extend(logs_r)

    # Actor-Critic
    env = CartPoleWrapper(seed=seed)
    logs_ac = run_actor_critic(seed, env, cfg, lr_pi=1e-3, lr_v=3e-3)
    all_logs.extend(logs_ac)

    # DQN (Double DQN)
    env = CartPoleWrapper(seed=seed)
    dqn_cfg = DQNConfig(double_dqn=True)
    logs_dqn = run_dqn(seed, env, cfg, dqn_cfg)
    all_logs.extend(logs_dqn)

# Save logs
log_path = os.path.join(LOGDIR, 'cartpole_logs.jsonl')
with open(log_path, 'w') as f:
    for row in all_logs:
        f.write(json.dumps(row) + '\n')
print(f'\nSaved {len(all_logs)} log entries to {log_path}')

## 3. Learning Curves

### Individual Algorithm Curves

In [ ]:
# Load and organize logs
rows = []
with open(log_path) as f:
    for line in f:
        rows.append(json.loads(line))

by_algo = defaultdict(lambda: defaultdict(list))
for r in rows:
    by_algo[r['algo']][r['steps']].append(r['return_mean'])

colors = {'REINFORCE': 'tab:blue', 'ActorCritic': 'tab:green', 'DQN': 'tab:red'}

def moving_avg(y, k=5):
    if k <= 1: return y
    kernel = np.ones(k) / k
    return np.convolve(y, kernel, mode='same')

# Individual learning curves per algorithm
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (algo, color) in zip(axes, colors.items()):
    if algo not in by_algo:
        continue
    data = by_algo[algo]
    xs = np.array(sorted(data.keys()))
    ys = np.array([np.mean(data[s]) for s in xs])
    q25 = np.array([np.percentile(data[s], 25) for s in xs])
    q75 = np.array([np.percentile(data[s], 75) for s in xs])
    ax.plot(xs, moving_avg(ys), color=color, lw=2)
    ax.fill_between(xs, q25, q75, alpha=0.2, color=color)
    ax.axhline(475, color='gray', ls='--', alpha=0.5, label='Solved (475)')
    ax.set_xlabel('Environment Steps')
    ax.set_ylabel('Mean Return')
    ax.set_title(algo)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right')

plt.suptitle('Individual Learning Curves (mean ± IQR across seeds)', fontsize=14)
plt.tight_layout(rect=[0,0,1,0.93])
plt.show()

### Combined Comparison

In [ ]:
plt.figure(figsize=(10, 6))
for algo, color in colors.items():
    if algo not in by_algo:
        continue
    data = by_algo[algo]
    xs = np.array(sorted(data.keys()))
    ys = np.array([np.mean(data[s]) for s in xs])
    q25 = np.array([np.percentile(data[s], 25) for s in xs])
    q75 = np.array([np.percentile(data[s], 75) for s in xs])
    plt.plot(xs, moving_avg(ys), color=color, lw=2, label=algo)
    plt.fill_between(xs, q25, q75, alpha=0.15, color=color)

plt.axhline(475, color='gray', ls='--', alpha=0.5, label='Solved (475)')
plt.xlabel('Environment Steps')
plt.ylabel('Mean Return')
plt.title('DQN vs REINFORCE vs Actor-Critic on CartPole-v1')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Summary Statistics

In [ ]:
def solved(returns, threshold=475):
    return any(r >= threshold for r in returns)

def auc(data):
    xs = np.array(sorted(data.keys()))
    ys = np.array([np.mean(data[s]) for s in xs])
    return float(np.trapz(ys, xs))

print('Algorithm Performance Summary')
print('=' * 65)
print(f'{"Algorithm":15s} {"Final Return":>14s} {"AUC":>12s} {"Solved?":>8s}')
print('-' * 65)

for algo in colors:
    if algo not in by_algo:
        continue
    data = by_algo[algo]
    xs = sorted(data.keys())
    final_returns = data[xs[-1]]
    all_returns = [v for s in xs for v in data[s]]
    area = auc(data)
    is_solved = solved(all_returns)
    print(f'{algo:15s} {np.mean(final_returns):>10.1f} ± {np.std(final_returns):>3.0f} {area:>12,.0f} {"  ✓" if is_solved else "  ✗":>8s}')

### Final Performance Bar Chart

In [ ]:
algos = [a for a in colors if a in by_algo]
final_means = []
final_stds = []
aucs = []

for algo in algos:
    data = by_algo[algo]
    xs = sorted(data.keys())
    final = data[xs[-1]]
    final_means.append(np.mean(final))
    final_stds.append(np.std(final))
    aucs.append(auc(data))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Final return
bars = ax1.bar(algos, final_means, yerr=final_stds, capsize=5,
               color=[colors[a] for a in algos], alpha=0.8)
ax1.axhline(475, color='gray', ls='--', alpha=0.5)
ax1.set_ylabel('Mean Return')
ax1.set_title('Final Performance (last evaluation)')
ax1.grid(axis='y', alpha=0.3)

# AUC (sample efficiency)
ax2.bar(algos, aucs, color=[colors[a] for a in algos], alpha=0.8)
ax2.set_ylabel('Area Under Curve')
ax2.set_title('Sample Efficiency (AUC)')
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Algorithm Comparison Summary', fontsize=14)
plt.tight_layout(rect=[0,0,1,0.93])
plt.show()

## Key Takeaways

DQN benefits from experience replay and target networks, giving it
stable learning with good sample efficiency. REINFORCE has high variance
due to Monte Carlo returns but can converge given enough episodes.
Actor-Critic combines the best of both worlds: policy gradient updates
with a learned baseline that reduces variance. All three algorithms
can solve CartPole-v1, but they differ in convergence speed and
stability across seeds.